# 03 · Tables for a dashboard

Each cell builds one small table and saves it as a CSV in `dashboard_out/`, ready to chart in Excel,
a BI tool, or a plotting notebook.

In [1]:
import os
from silver import load_clean

df = load_clean()
after_hours_phone = df[df["is_telephone"] & df["is_after_hours"]]

os.makedirs("dashboard_out", exist_ok=True)

### Headline numbers

In [2]:
import pandas as pd
after_hours = df[df["is_after_hours"]]
kpis = pd.DataFrame([{
    "after_hours_telephone_appts": len(after_hours_phone),
    "telephone_no_show_%": round(after_hours_phone["is_noshow"].mean() * 100, 1),
    "in_person_no_show_%": round(after_hours[~after_hours["is_telephone"]]["is_noshow"].mean() * 100, 1),
    "telephone_share_of_after_hours_%": round(len(after_hours_phone) / len(after_hours) * 100, 1),
}])
kpis.to_csv("dashboard_out/kpis.csv", index=False)
kpis

,after_hours_telephone_appts,telephone_no_show_%,in_person_no_show_%,telephone_share_of_after_hours_%
0,4383,19.4,10.0,61.7


### Demand by day and hour (for a heatmap)

In [3]:
demand = after_hours_phone.groupby(["day_of_week", "appt_hour"])["is_completed"].sum()
demand = demand.reset_index(name="appointments")
demand.to_csv("dashboard_out/demand_by_day_hour.csv", index=False)
demand.head()

,day_of_week,appt_hour,appointments
0,Friday,17,138
1,Friday,18,237
2,Friday,19,200
3,Friday,20,4
4,Monday,17,187


### No-show rate by how far ahead it was booked

In [4]:
def lead_bucket(minutes):
    if minutes < 30:  return "under 30 min"
    if minutes < 60:  return "30-60 min"
    if minutes < 120: return "1-2 hours"
    if minutes < 240: return "2-4 hours"
    return "4+ hours"

phone = after_hours_phone.copy()
phone["booked_ahead"] = phone["booking_lead_min"].apply(lead_bucket)
lead = phone.groupby("booked_ahead").agg(
    appointments=("is_noshow", "size"),
    no_show_pct=("is_noshow", "mean"),
)
lead["no_show_pct"] = (lead["no_show_pct"] * 100).round(1)
lead.to_csv("dashboard_out/no_show_by_booking_lead.csv")
lead

,appointments,no_show_pct
booked_ahead,,
1-2 hours,1445,15.8
2-4 hours,483,16.6
30-60 min,1984,21.8
4+ hours,65,18.5
under 30 min,406,23.6


### Telephone share and no-show by month (for a trend line)

In [5]:
after_hours = df[df["is_after_hours"]]
monthly = after_hours.groupby("appt_month").agg(
    after_hours_total=("is_telephone", "size"),
    telephone=("is_telephone", "sum"),
)
monthly["telephone_share_%"] = (monthly["telephone"] / monthly["after_hours_total"] * 100).round(1)
phone_monthly = after_hours_phone.groupby("appt_month")["is_noshow"].mean() * 100
monthly["telephone_no_show_%"] = phone_monthly.round(1)
monthly.to_csv("dashboard_out/monthly_trend.csv")
monthly

,after_hours_total,telephone,telephone_share_%,telephone_no_show_%
appt_month,,,,
2009-12,1029,597,58.0,8.4
2010-01,1124,697,62.0,22.8
2010-02,1158,718,62.0,19.5
2010-03,1212,770,63.5,21.6
2010-04,1116,698,62.5,21.2
2010-05,1062,650,61.2,18.2
2010-06,401,253,63.1,26.9


### No-show rate by physician (coaching targets)

In [6]:
by_provider = after_hours_phone.groupby("PROVIDER_ID").agg(
    appointments=("is_noshow", "size"),
    no_show_pct=("is_noshow", "mean"),
)
by_provider["no_show_pct"] = (by_provider["no_show_pct"] * 100).round(1)
by_provider = by_provider[by_provider["appointments"] >= 50].sort_values("no_show_pct", ascending=False)
by_provider.to_csv("dashboard_out/no_show_by_provider.csv")
by_provider.head()

,appointments,no_show_pct
PROVIDER_ID,,
42133974,242,43.0
15412841,68,30.9
79864737,66,30.3
57772125,70,27.1
25326992,65,26.2
